# Daily Challenge — MCP Ping-Pong

**Week 10 · Day 1 — Bootcamp GenAI & Machine Learning 2026**

Build a minimal MCP server and client (no LLMs) that talk over STDIO, expose a tiny weather tool, and observe data flowing end-to-end.

| File | Role |
|------|------|
| `server.py` | FastMCP `WeatherDemo` — 1 tool + 1 resource |
| `client.py` | Spawns server, discovers capabilities, invokes tool |

In [ ]:
%pip install -q "mcp[cli]"

---
## server.py — WeatherDemo

- **Tool** `get_weather(city)` — static lookup for Paris, London, NYC, Tokyo, Sydney
- **Resource** `cities://list` — concrete URI returning supported cities
- Logs to stderr so tool calls are visible during debugging

In [ ]:
%%writefile server.py
import logging
import sys
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr,
                    format="[SERVER] %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

mcp = FastMCP("WeatherDemo")

WEATHER_DB = {
    "paris":  {"city": "Paris",  "temp_c": 21, "condition": "sunny"},
    "london": {"city": "London", "temp_c": 14, "condition": "cloudy"},
    "nyc":    {"city": "NYC",    "temp_c": 18, "condition": "partly cloudy"},
    "tokyo":  {"city": "Tokyo",  "temp_c": 26, "condition": "humid"},
    "sydney": {"city": "Sydney", "temp_c": 22, "condition": "clear"},
}


@mcp.tool()
def get_weather(city: str) -> dict:
    """Returns static weather data for a supported city."""
    logger.info(f"get_weather called → city={city!r}")
    data = WEATHER_DB.get(city.lower())
    if data:
        return data
    supported = ", ".join(c.title() for c in WEATHER_DB)
    return {"error": f"City '{city}' not found. Supported: {supported}"}


@mcp.resource("cities://list")
def list_cities() -> str:
    """Returns a newline-separated list of supported cities."""
    logger.info("cities://list resource read")
    return "\n".join(c.title() for c in WEATHER_DB)


if __name__ == "__main__":
    mcp.run()

---
## client.py

1. Spawns server over STDIO
2. Lists resources + tools
3. Reads `cities://list`
4. Calls `get_weather('Paris')` and `get_weather('Atlantis')` (error case)

In [ ]:
%%writefile client.py
import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:

            await session.initialize()
            print("Session initialized ✅\n")

            # List resources
            resources = await session.list_resources()
            print("=== Resources ===")
            for r in resources.resources:
                print(f"  - {r.name}: {r.uri}")

            # List tools
            tools = await session.list_tools()
            print("\n=== Tools ===")
            for t in tools.tools:
                print(f"  - {t.name}: {t.description}")

            # Read cities://list
            content = await session.read_resource("cities://list")
            print("\n=== cities://list ===")
            print(content.contents[0].text)

            # Call get_weather for Paris
            result = await session.call_tool("get_weather", {"city": "Paris"})
            print("=== get_weather('Paris') ===")
            try:
                print(json.dumps(json.loads(result.content[0].text), indent=2))
            except Exception:
                print(result.content[0].text)

            # Error case
            result_err = await session.call_tool("get_weather", {"city": "Atlantis"})
            print("\n=== get_weather('Atlantis') — error handling ===")
            print(result_err.content[0].text)


if __name__ == "__main__":
    asyncio.run(run())

---
## Run

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "client.py"],
    capture_output=True, text=True, timeout=30
)
print(result.stdout)

---
## Terminal Output

```
Session initialized ✅

=== Resources ===
  - list_cities: cities://list

=== Tools ===
  - get_weather: Returns static weather data for a supported city.

=== cities://list ===
Paris
London
Nyc
Tokyo
Sydney

=== get_weather('Paris') ===
{
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}

=== get_weather('Atlantis') — error handling ===
{"error": "City 'Atlantis' not found. Supported: Paris, London, Nyc, Tokyo, Sydney"}
```

**Key observations:**
- `cities://list` is a **concrete resource** (no `{}` placeholders) → appears in `list_resources()`
- Tool returns a `dict` → MCP serializes it to JSON automatically
- Unknown city returns an error dict instead of raising an exception — graceful degradation
- STDIO transport: zero config, server spawned as subprocess, dies when client disconnects

---
## Tool vs Resource — Key Difference

| | Tool | Resource |
|-|------|----------|
| **Purpose** | Action / computation | Read-only context / data |
| **Args** | Typed parameters | URI (+ template vars) |
| **Side effects** | Allowed | Not expected |
| **Example** | `get_weather(city)` | `cities://list` |
| **LLM use** | LLM decides when to call | LLM uses as background context |